# Osprey-Optimized GNN for WSN Intrusion Detection — Walkthrough

This notebook runs the whole project end-to-end and explains each stage as it goes:

1. **Generate** a synthetic WSN-DS dataset (or load the real one)
2. **Build** the communication graph for each LEACH round
3. **Optimize** the GNN's hyper-parameters with the **Osprey Optimization Algorithm**
4. **Train** the best model and **compare** it against baselines

> Run from the repository root. Uses tiny "quick" budgets so it finishes in ~1-2 min on CPU.


In [ ]:
import os, sys
# ensure repo root on path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt

from config import DataConfig, OspreyConfig, TrainConfig, apply_quick, CLASS_NAMES, SEARCH_SPACE
from src.utils import set_seed

set_seed(42)
data_cfg, osp_cfg, tr_cfg = DataConfig(), OspreyConfig(), TrainConfig()
apply_quick(data_cfg, osp_cfg, tr_cfg)   # small budgets for a fast demo
print("Classes:", CLASS_NAMES)


## 1. Generate the dataset

Each row is one sensor node in one LEACH round. The generator encodes the *behavioural
signature* of each attack (e.g. a blackhole receives lots of data but forwards almost none).

In [ ]:
from src.data.synthetic import generate_and_save
df = generate_and_save(data_cfg, seed=42)
print(df.shape)
df.head()


In [ ]:
# class balance + the tell-tale forward ratio per class
import pandas as pd
summary = df.groupby(df["label"].map(dict(enumerate(CLASS_NAMES)))).agg(
    count=("label", "size"),
    mean_forward_ratio=("Forward_Ratio", "mean"),
    mean_adv_s=("ADV_S", "mean"),
    mean_energy=("Consumed_Energy", "mean"),
)
summary


Notice the signatures: **Blackhole/Grayhole** have low `Forward_Ratio`, **Flooding** has a
huge `ADV_S` and energy, etc. These are exactly the patterns the model must learn.

## 2. Build the graph

Every round becomes a graph: members link to their Cluster Head, CHs interlink, and each node
links to its spatial k-nearest neighbours. All rounds are packed into one disjoint-union graph.

In [ ]:
from src.data.loader import load_dataset
from src.graph.builder import build_graph

ds = load_dataset(data_cfg.csv_path, seed=42)
graph = build_graph(ds, knn_k=data_cfg.knn_k, seed=42)
print(f"nodes={graph.num_nodes}  edges={graph.edge_index.shape[1]}  features={graph.num_features}")
print(f"train={graph.train_mask.sum()}  val={graph.val_mask.sum()}  test={graph.test_mask.sum()}")


In [ ]:
# visualise one round's topology
import networkx as nx
round0 = ds.round_id == np.unique(ds.round_id)[0]
idx = np.where(round0)[0]
G = nx.Graph()
local = {g: i for i, g in enumerate(idx)}
for g in idx:
    G.add_node(local[g])
ei = graph.edge_index
for s, d in zip(ei[0], ei[1]):
    if s in local and d in local and s != d:
        G.add_edge(local[s], local[d])
pos = {local[g]: ds.pos[g] for g in idx}
colors = [ds.y[g] for g in idx]
plt.figure(figsize=(7,6))
nx.draw(G, pos, node_color=colors, cmap="tab10", node_size=60, edge_color="#cccccc", width=0.5)
plt.title("One LEACH round — node colour = class")
plt.show()


## 3. Osprey Optimization of the GNN

The search space (`config.SEARCH_SPACE`) covers layer type, depth, width, dropout, learning
rate and weight decay. The Osprey algorithm maximises **validation macro-F1**.

In [ ]:
for k, v in SEARCH_SPACE.items():
    print(f"  {k:12s} range {v}")


In [ ]:
from src.optimize import run_osprey_search
best_hp, result = run_osprey_search(graph, osp_cfg, seed=42)
print("\nBest validation macro-F1:", round(result.best_fitness, 4))
print("Best hyper-parameters:", best_hp)


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(range(1, len(result.history)+1), result.history, "o-", label="best-so-far")
plt.plot(range(1, len(result.mean_history)+1), result.mean_history, "s--", alpha=0.6, label="population mean")
plt.xlabel("Iteration"); plt.ylabel("Validation macro-F1")
plt.title("Osprey convergence"); plt.legend(); plt.grid(alpha=0.3); plt.show()


## 4. Final model & baseline comparison

Retrain the winning config to convergence and compare against a Random Forest, a graph-blind
MLP, and an untuned GNN.

In [ ]:
from src.evaluate import run_final_evaluation
summary = run_final_evaluation(graph, best_hp, tr_cfg, osprey_result=result, seed=42, save_model=False)
import pandas as pd
pd.DataFrame(summary["results"]).T[["accuracy", "macro_f1", "macro_recall"]].round(3)


In [ ]:
from PIL import Image
for name in ["confusion_matrix.png", "model_comparison.png"]:
    p = os.path.join("results", name)
    if os.path.exists(p):
        plt.figure(figsize=(7,5)); plt.imshow(Image.open(p)); plt.axis("off"); plt.show()


## Takeaways

- The **Osprey Optimization Algorithm** automatically discovers a strong GNN configuration,
  substantially improving on an untuned GNN.
- Modelling the WSN as a **graph** lets the detector reason about a node *relative to its
  neighbours* — the natural way to spot routing attacks.
- Everything is reproducible: rerun with `python main.py --all` (or `--quick`).

See `docs/methodology.md` and `docs/osprey_algorithm.md` for the full explanation.